## Initializing the functions

In [0]:
%run /Configurations/Init_Scripts

## Declaring the source and destination paths

In [0]:
import requests 
import json

In [0]:
dbutils.widgets.text("PromotionName", "4P Beta")
PromotionName = dbutils.widgets.get("PromotionName")

dbutils.widgets.text("job_id","-1")
JobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("run_id","-1")
PipelineRunId = dbutils.widgets.get("run_id")

ConfigID=dbutils.widgets.text("ConfigID","39")
ConfigID=dbutils.widgets.get("ConfigID")

dbutils.widgets.text('sourceTypeId','13')
sourceTypeId = dbutils.widgets.get('sourceTypeId')

dbutils.widgets.text('RecordTypeId','0123h000000khU7AAI')
RecordTypeId = dbutils.widgets.get('RecordTypeId')

dbutils.widgets.text('CreatedBy','ADB_MoxieInvoiceExceptionFullLoad')
CreatedBy = dbutils.widgets.get('CreatedBy')


dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
RawPath_InvoiceException=ExternalLocation_raw+'/SalesForce/Moxie/InvoiceException'

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")


In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
# RawPath_InvoiceException = '/mnt/raw/SalesForce/Moxie/InvoiceException'


API_RETRY_LIMIT = 5

raw_schema = StructType([
        StructField("Id", StringType(), True),
        StructField("SAP_Account_Id__c", StringType(), True),
        StructField("Patient_Classification__c", StringType(), True),
        StructField("Elite_Treatment_Visit_Date__c", DateType(), True),
        StructField("Elite_Cycle__c", IntegerType(), True),
        StructField("Comments", StringType(), True),
        StructField("CaseNumber", StringType(), True),
        StructField("Phone_Number__c", StringType(), True),
        StructField("Incremental_Invoice_Charge__c", FloatType(), True),
        StructField("CaseOwner__c", StringType(), True),
        StructField("Other_Type__c", StringType(), True),
    ])

In [0]:
#Authorization
Client_id = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-ClientID')
Client_secret = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Secret')
Username = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-UserName')
Password = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Password')
InstanceURL = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-InstanceURL')
Domain   = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Domain')

## Data Ingestion

In [0]:
#Generate Access Token
def generate_token():   
    token_endpoint = f'https://{Domain}/services/oauth2/token' 
    payload = {
        'grant_type': 'password',
        'client_id': Client_id,
        'client_secret': Client_secret,
        'username': Username,
        'password': Password
    }

    retry_count = 0
    while (retry_count <= API_RETRY_LIMIT):
      response = requests.post(token_endpoint, data=payload)
      if(response.status_code == 200): 
          print(f"Successfully retrieved the Token.")
          response_data = response.json()
          return response_data
      elif(500 <= response.status_code <= 599 or response.status_code == 429):
        print(f'url:{url};', f'response_code:{response.status_code};' f'error_message:{response.text}')
        print("Waiting for 90 seconds...")
        sleep(90)
        retry_count += 1
      else:
        raise Exception(f'url:{endpoint};', f'response_code:{response.status_code};' f'error_message:{response.text}')
    raise Exception(f'url:{endpoint};', f'response_code:{response.status_code};' f'error_message:{response.text}')

In [0]:
#Reading Data from Moxie
def Query(soql_query):
    endpoint = f"https://{InstanceURL}/services/data/v58.0/query/"
    records = []

    headers = {
            'Authorization': f'Bearer {access_token}',
            'Content-Type': 'application/json'
        }

    retry_count = 0
    while (retry_count <= API_RETRY_LIMIT):
      response = requests.get(endpoint, headers=headers, params={'q': f'{soql_query}'})
      if(response.status_code == 200): 
          print(f"Successfully retrieved the Data.")
          response_data = response.json()
          return response_data
      elif(500 <= response.status_code <= 599 or response.status_code == 429):
        print(f'url:{url};', f'response_code:{response.status_code};' f'error_message:{response.text}')
        print("Waiting for 90 seconds...")
        sleep(90)
        retry_count += 1
      else:
        raise Exception(f'url:{endpoint};', f'response_code:{response.status_code};' f'error_message:{response.text}')
    raise Exception(f'url:{endpoint};', f'response_code:{response.status_code};' f'error_message:{response.text}')

In [0]:
log_dict = {
 "ConfigID" : ConfigID,
  "SourceTypeID" : sourceTypeId,
  "Source" : "Moxie",
  "SourceFileName" : "InvoiceException",
  "Destination" : "rawzone.raw_InvoiceException",
  "DestinationFileName" : "",
  "Run_ID": PipelineRunId,
  "Job_ID": JobId
}
#print(log_dict)
audit_df = spark.createDataFrame([log_dict])
display(audit_df)

## Pulling Data from Moxie

In [0]:
try:
    #logIntoPromotionLogTable(audit_df,CreatedBy,"InProgress")
    access_token = generate_token()['access_token']
    print(access_token)

    raw_data = Query(f'''Select 
                        Id,
                        SAP_Account_Id__c,
                        Patient_Classification__c,
                        Elite_Treatment_Visit_Date__c,
                        Elite_Cycle__c,
                        Comments,
                        CaseNumber,
                        Phone_Number__c,
                        Incremental_Invoice_Charge__c,
                        CaseOwner__c,
                        Other_Type__c
                        from case 
                        where recordtypeid = \'{RecordTypeId}\' and Sub_Type__c = \'Exception Credit\'
    ''')

    raw_df = spark.createDataFrame(raw_data['records'],raw_schema)\
          .drop("attributes")
    raw_df.display()

    exception_df = raw_df.withColumnRenamed("SAP_Account_Id__c","ShipToAccountId")\
                    .withColumnRenamed("Phone_Number__c","PhoneNumber")\
                    .withColumnRenamed("Elite_Treatment_Visit_Date__c","CycleDate")\
                    .withColumnRenamed("Elite_Cycle__c","CycleCount")\
                    .withColumnRenamed("Incremental_Invoice_Charge__c","IncrementalInvoiceCharge")\
                    .withColumnRenamed("Patient_Classification__c","PatientClassificationName")\
                    .withColumnRenamed("CaseNumber","MoxieCaseNumber")\
                    .withColumnRenamed("CaseOwner__c","CaseOwner")\
                    .withColumnRenamed("Id","MoxieCaseID")\
                    .withColumnRenamed("Other_Type__c","SalesOrderNumber")\
                    .withColumn('CreatedBy',lit(CreatedBy))\
                    .withColumn('CreatedBy',lit(CreatedBy))\
                    .withColumn('UpdatedBy',lit(CreatedBy))\
                    .withColumn("Createddate", current_timestamp())\
                    .withColumn("Updateddate", current_timestamp())\
                    .withColumn("Active", lit('1').cast('Int'))\
                    .select('MoxieCaseID','PhoneNumber','ShipToAccountID','PatientClassificationName','CycleDate',col('CycleCount').cast('String'),col('IncrementalInvoiceCharge').cast('string'),'SalesOrderNumber','MoxieCaseNumber','CaseOwner','Comments','CreatedDate','CreatedBy','UpdatedDate','UpdatedBy')

    exception_df.write.format("delta").mode("append").save(RawPath_InvoiceException)
    logIntoPromotionLogTable(audit_df,CreatedBy,"Succeeded")
except Exception as e:
    logIntoPromotionLogTable(audit_df,CreatedBy,"Failed",str(e))
    raise e